In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Example inputs
job_description = """
Experienced Python developer with knowledge of machine learning,
data analysis, and deep learning. Worked with TensorFlow and pandas.
"""

resume  = """ machine learning engineer with experience in Python and data analysis  developer.
"""

# Step 1: Convert text to vectors
vectorizer = TfidfVectorizer(stop_words='english')
vectors = vectorizer.fit_transform([resume, job_description])

# Step 2: Compute similarity
similarity = cosine_similarity(vectors[0], vectors[1])[0][0]

# Step 3: Convert to percentage
score = round(similarity * 100, 2)

print(f"Match Score: {score}%")

Match Score: 48.6%


this modal is trying to match all the texts indivully .

In [2]:
skills_list = [
    "python", "java", "c++",
    "machine learning", "deep learning",
    "tensorflow", "pytorch",
    "data analysis", "pandas", "numpy",
    "sql", "nlp", "computer vision"
]
# Example inputs
job_description = """
Experienced Python developer with knowledge of machine learning,
data analysis, and deep learning. Worked with TensorFlow and pandas.
"""

resume = """ machine learning  with experience in Python and data analysis  developer.
"""
#skill extraction
import re

def extract_skills(text, skills_list):
    text = text.lower()
    found_skills = []

    for skill in skills_list:
        # simple match
        if re.search(r'\b' + re.escape(skill) + r'\b', text):
            found_skills.append(skill)

    return found_skills

resume_skills = extract_skills(resume, skills_list)
job_skills = extract_skills(job_description, skills_list)


missing_skills = list(set(job_skills) - set(resume_skills))
print("Missing Skills:", missing_skills)

matching_score = 100*(len(job_skills))/(len(job_skills)+len(missing_skills))
print("Matching Score:", matching_score)

Missing Skills: ['deep learning', 'pandas', 'tensorflow']
Matching Score: 66.66666666666667


still very limited . It doesn't have understanding of words or sentences .

so we will use modern Embeddings .

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Skills list
skills_list = [
    "python", "machine learning", "deep learning",
    "tensorflow", "pytorch", "data analysis",
    "sql", "nlp", "computer vision"
]

# Example text
job_description = """
Experienced Python developer with knowledge of machine learning,
data analysis, and deep learning. Worked with TensorFlow and pandas.
"""

resume  = """ ML engineer with experience in Python and data analysis  developer.
"""

# Encode everything
skill_embeddings = model.encode(skills_list)
resume_embedding = model.encode([resume])
jd_embedding = model.encode([job_description])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import re

# Map common short forms in resumes/JDs to the canonical skill names.
ALIASES = {
    "ml": "machine learning",
    "dl": "deep learning",
}
def split_sentences(text):
    sentences = re.split(r'[.,\n]+', text.lower())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

def extract_skills_semantic(text, skills_list, threshold=0.5):
    sentences = split_sentences(text)

    if not sentences:   # <-- safety
        return []

    sentence_embeddings = model.encode(sentences)

    found_skills = []

    for skill in skills_list:
        skill_embedding = model.encode([skill])[0]

        for sent_emb in sentence_embeddings:
            sim = cosine_similarity(
                [sent_emb],
                [skill_embedding]
            )[0][0]

            if sim > threshold:
                found_skills.append(skill)
                break

    return found_skills
# Example text
job_description = """
Experienced Python developer with knowledge of machine learning,
data analysis, and deep learning. Worked with TensorFlow and pandas.
"""

resume  = """ ML engineer with experience in Python and data analysis  developer.
"""       
       
resume_skills = extract_skills_semantic(resume, skills_list)
jd_skills = extract_skills_semantic(job_description, skills_list)

print("Resume Skills:", resume_skills)
print("JD Skills:", jd_skills)

# Skills present in the JD but missing from the resume.
missing_skills = list(set(jd_skills) - set(resume_skills))
print("Missing Skills:", missing_skills)
#print("Matching_score:",similarity(resume, job_description))

Resume Skills: ['python']
JD Skills: ['python', 'machine learning', 'deep learning', 'tensorflow', 'pytorch', 'data analysis', 'sql', 'computer vision']
Missing Skills: ['deep learning', 'data analysis', 'sql', 'tensorflow', 'computer vision', 'machine learning', 'pytorch']


this embedding method used for skill extraction and finding similairty is even giving poorer results. so we will use hybrid system.

In [23]:
def split_sentences(text):
    sentences = re.split(r'[.,\n]+', text.lower())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences
print(split_sentences(resume))
print(split_sentences(job_description))

['ml engineer with experience in python and data analysis  developer']
['experienced python developer with knowledge of machine learning', 'data analysis', 'and deep learning', 'worked with tensorflow and pandas']


In [17]:


def calculate_similarity(resume_text, job_description):
    embeddings = model.encode([resume_text, job_description])

    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return round(similarity * 100, 2)


def extract_skills(text, skills_list):
    text = text.lower()
    found_skills = []

    for skill in skills_list:
        if skill in text:
            found_skills.append(skill)

    return found_skills


score = calculate_similarity(resume, job_description)
print("Matching_score:",similarity(resume, job_description))
resume_skills = extract_skills(resume, skills_list)
job_skills = extract_skills(job_description, skills_list)

print("Resume Skills:", resume_skills)
print("JD Skills:", jd_skills)

missing_skills = list(set(job_skills) - set(resume_skills))
print("Missing Skills:", missing_skills)


Matching_score: [[0.76651734]]
Resume Skills: ['python', 'data analysis']
JD Skills: ['python', 'machine learning', 'deep learning', 'tensorflow', 'pytorch']
Missing Skills: ['deep learning', 'machine learning', 'tensorflow']


In [ ]:
def extract_skills_hybrid(text, skills_list, threshold=0.5):
    text_lower = text.lower()
    
    # 1. Exact match
    exact_skills = [skill for skill in skills_list if skill in text_lower]

    # 2. Semantic match
    sentences = split_sentences(text)
    
    if not sentences:
        return exact_skills

    sentence_embeddings = model.encode(sentences)

    semantic_skills = []

    for skill in skills_list:
        if skill in exact_skills:
            continue  # already found

        skill_embedding = model.encode([skill])[0]

        for sent_emb in sentence_embeddings:
            sim = cosine_similarity(
                [sent_emb],
                [skill_embedding]
            )[0][0]

            if sim > threshold:
                semantic_skills.append(skill)
                break

    # Combine both
    final_skills = list(set(exact_skills + semantic_skills))

    return final_skills

resume_skills = extract_skills_hybrid(resume, skills_list)
job_skills = extract_skills_hybrid(job_description, skills_list)

print("Resume Skills:", resume_skills)
print("JD Skills:", job_skills)


Resume Skills: ['data analysis', 'python']
JD Skills: ['deep learning', 'data analysis', 'sql', 'tensorflow', 'computer vision', 'machine learning', 'pytorch', 'python']


In [ ]:
text_lower = resume.lower()
[skill for skill in skills_list if skill in text_lower]

['python', 'data analysis']

In [40]:
text_lower = job_description.lower()
[skill for skill in skills_list if skill in text_lower]

['python', 'machine learning', 'deep learning', 'tensorflow', 'data analysis']

with exact matching data analysis , pandas in missing

In [49]:
def skill_match_score(resume_skills, job_skills):
    if not job_skills:
        return 0

    matched = len(set(resume_skills) & set(job_skills))
    total = len(job_skills)

    return (matched / total) * 100

def final_score(resume, job):
    sim_score = calculate_similarity(resume, job)

    resume_skills = extract_skills_hybrid(resume, skills_list)
    job_skills = extract_skills(job, skills_list)

    skill_score = skill_match_score(resume_skills, job_skills)

    final = 0.6 * sim_score + 0.4 * skill_score

    return {
        "semantic_score": sim_score,
        "skill_score": skill_score,
        "final_score": round(final, 2),
        "missing_skills": list(set(job_skills) - set(resume_skills))
    }

In [50]:
final_score(resume, job_description)

{'semantic_score': np.float32(76.65),
 'skill_score': 40.0,
 'final_score': np.float32(61.99),
 'missing_skills': ['deep learning', 'machine learning', 'tensorflow']}